In [ ]:
# Script d'exportation de graphe egocentré profondeur 2 pour chaque individu du graphe en utilisant sparqlwrqpper
from SPARQLWrapper import SPARQLWrapper
from SPARQLWrapper import JSON
import matplotlib.pyplot as plt
from SPARQLWrapper import TURTLE
from rdflib import Graph
from utils import show_graph
from utils import save_graph_html
import os
import importlib
import utils 
importlib.reload(utils)
import time
import networkx as nx
import csv
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from shapely import wkt

In [ ]:
# set it uppppppp
endpoint = SPARQLWrapper(" ")
endpoint.setReturnFormat(JSON)
output_dir = "./cartes_ego"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
#requete pour avoir la liste d'agents
query_liste_agents = """
PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?agent ?roleLabel
WHERE {
    ?s a rico:Record .
    ?rel rico:relationHasSource ?s .  
    ?rel rico:relationHasTarget ?agent .
    OPTIONAL { 
        ?rel rico:withCreationRole ?role . 
        ?role skos:prefLabel ?roleLabel .
    }
}
"""

In [ ]:
endpoint.setQuery(query_liste_agents)
# Exécution de la première requête et extraction de la liste des URIs
resultats_json = endpoint.queryAndConvert()
liste_agents = []
for row in resultats_json["results"]["bindings"]:
    agent_uri = row["agent"]["value"]
    # Gestion du cas où le rôle n'est pas renseigné (valeur par défaut : "Inconnu")
    role_label = row["roleLabel"]["value"] if "roleLabel" in row else "Inconnu"
    liste_agents.append({"uri": agent_uri, "role": role_label})

print(f"{len(liste_agents)} agents à traiter.\n")

print("Lancement du traitement")

In [ ]:
def save_map_html(df, output_filename):
    # 1. On ne garde que les agents qui ont une géométrie
    df_geo = df.dropna(subset=['wkt']).copy()
    if df_geo.empty:
        print(f"Aucune géométrie trouvée pour {output_filename}")
        return

    # 2. Conversion du WKT en géométrie réelle
    df_geo['geometry'] = df_geo['wkt'].apply(wkt.loads)
    
    # 3. Création du GeoDataFrame en précisant le CRS source (Lambert 93)
    gdf = gpd.GeoDataFrame(df_geo, geometry='geometry', crs="EPSG:2154")
    
    # 4. Conversion cruciale vers WGS84 (lat/lon) pour la carte web
    gdf_web = gdf.to_crs(epsg=4326)
    
    # 5. Création de la carte
    m = folium.Map(location=[48.8566, 2.3522], zoom_start=12)
    
    # 6. Ajout des marqueurs avec les coordonnées converties
    for _, row in gdf_web.iterrows():
        # row['geometry'].y est la latitude, row['geometry'].x est la longitude
        folium.Marker(
            location=[row['geometry'].y, row['geometry'].x],
            popup=row['nom'],
            tooltip=row['nom']
        ).add_to(m)
        
    m.save(output_filename)

In [ ]:
def process_carto_reseau(liste_a_traiter, role_cible=None):
    endpoint.setReturnFormat(TURTLE)
    for agent_data in liste_a_traiter:
        agent_uri = agent_data["uri"]
        role_label = agent_data["role"]
                
        if role_cible is not None and role_label.lower() != role_cible.lower():
            continue
        
        
        nom_court = agent_uri.split('/')[-1]
        if nom_court.lower() == "anonyme":
            continue

        print(f"\n--> Traitement de l'agent : {nom_court}, URI : {agent_uri}")
        query_ego_2 = f"""
            PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
            PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
            PREFIX adb: <http://data.soduco.fr/def/annuaire#>
            PREFIX gsp: <http://www.opengis.net/ont/geosparql#>

            CONSTRUCT {{
            # NIVEAU 1 : Liens pour l'Agent principal (Ego)
            ?s ?labelRoleUri <{agent_uri}> ?adresseAgent ?geoAgent.
            
            # Sécurité si le rôle n'a pas de libellé textuel dans la base
            ?s rico:isRelatedTo <{agent_uri}> .
            
            <{agent_uri}> rico:name ?nomAgent .
            <{agent_uri}> adb:address ?adresseAgent .
            <{agent_uri}> adb:hasAddressGeometry ?geoAgent
            # --------------------------------------------------------------------------
            # NIVEAU 2 : Liens pour les Collaborateurs
            ?s ?labelOtherRoleUri ?otherAgent .
            ?s rico:isRelatedTo ?otherAgent .
            
            ?otherAgent rico:name ?otherNom .
            
            }}
            WHERE {{
            # 1. Ancrage sur l'Ego
            <{agent_uri}> rico:name ?nomAgent .
            ?rel rico:relationHasTarget <{agent_uri}> .
            ?rel rico:relationHasSource ?s .
            ?s a rico:Record .
            
            # Récupération du rôle de l'Ego et de son étiquette (ex: "Dessinateur")
            OPTIONAL {{ 
                ?rel rico:withCreationRole ?role . 
                OPTIONAL {{ ?role skos:prefLabel ?roleLabel . }}
            }}
            BIND(IF(BOUND(?roleLabel), IRI(CONCAT("http://localhost/role/", ?roleLabel)), ?role) AS ?labelRoleUri)

            # 2. Récupération des infos géographiques (via lien skos:exactMatch du mapping 2)
            OPTIONAL {{
            ?cluster skos:exactMatch <{agent_uri}> .
            OPTIONAL {{ ?cluster adb:address ?adresseAgent . }}
            OPTIONAL {{ ?cluster adb:hasAddressGeometry ?geoAgent . }}
                    }}

            # 3. Recherche des collaborateurs
            OPTIONAL {{
                ?otherRel rico:relationHasSource ?s .
                ?otherRel rico:relationHasTarget ?otherAgent .
                ?otherAgent rico:name ?otherNom .
                
                # Récupération du rôle du collaborateur et de son étiquette (ex: "Imprimeur")
                OPTIONAL {{ 
                    ?otherRel rico:withCreationRole ?otherRole . 
                    OPTIONAL {{ ?otherRole skos:prefLabel ?otherRoleLabel . }}
                }}
                
                BIND(IF(BOUND(?otherRoleLabel), IRI(CONCAT("http://localhost/role/", ?otherRoleLabel)), ?otherRole) AS ?labelOtherRoleUri)

                # Récupération infos géographiques pour le Collaborateur
            OPTIONAL {{
            ?clusterOther skos:exactMatch ?otherAgent .
            OPTIONAL {{ ?clusterOther adb:address ?adresseOther . }}
            OPTIONAL {{ ?clusterOther adb:hasAddressGeometry ?geoOther . }}
        }}
                
                FILTER(?otherAgent != <{agent_uri}>)
                }}
            }}  
        """
        endpoint.setQuery(query_ego_2)
        results = endpoint.queryAndConvert()
        
    
        # On initialise un graphe vide...
        g = Graph()
        g.parse(data=results, format="turtle")

        # 1. Requête SPARQL locale sur le graphe 'g' en mémoire
        # On extrait les agents collaborateurs, leurs noms et leurs géométries
        q_geo = """
        PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
        PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
        PREFIX adb: <http://data.soduco.fr/def/annuaire#>

        SELECT ?agent ?nom ?wkt
        WHERE {
            ?agent rico:name ?nom .
            OPTIONAL {
            ?cluster skos:exactMatch ?agent;
                adb:hasAddressGeometry ?wkt . }
        }
        """

        # 2. Exécution de la requête sur le graphe RDFLib
        results = g.query(q_geo)

        # 3. Conversion des résultats en liste de dictionnaires
        # On filtre les résultats pour ne garder que ceux qui ont une géométrie si nécessaire
        data = []
        for row in results:
            data.append({
                "agent": str(row.agent),
                "nom": str(row.nom),
                "wkt": str(row.wkt) if row.wkt else None
            })
            
        # 4. Création du DataFrame
        df = pd.DataFrame(data)
        
        # Maintenant, df contient vos colonnes 'agent', 'nom' et 'wkt'
        print(df.head()) # Vérification rapide

        # Enregsitrer graphe en html
        save_map_html(df,output_filename=f"./cartes_ego/carte_ego_{nom_court}.html")
        
        time.sleep(0.2)

In [ ]:
#execution

process_carto_reseau(liste_agents[0], portion="Dessinateur")
#process_carto_reseau(liste_agents, portion="Imprimeur")